# HCDE 530 — Week 5: App reviews exploration

This notebook answers five checklist questions using **`app_reviews_demo.csv`**. Run cells top to bottom (**Shift+Enter**).

**Kernel:** use your Week 5 environment (`.venv` / `week5/.venv`) so `pandas` is available.

## Setup — load the CSV

In [ ]:
import pandas as pd
from pathlib import Path

_paths = [Path("app_reviews_demo.csv"), Path("week5/app_reviews_demo.csv")]
_csv = next((p for p in _paths if p.is_file()), None)
if _csv is None:
    raise FileNotFoundError("Place app_reviews_demo.csv in week5/ or run from repo root.")

df = pd.read_csv(_csv)
print(f"Loaded: {_csv.resolve()}")
print(f"pandas {pd.__version__}")

---
## 1. What does your dataset look like? `head()`, `info()`

**Answer:** `head()` shows column names, example values, and types at a glance. `info()` lists every column, non-null counts, and dtypes — useful for seeing how many rows you really have per column.

In [ ]:
df.head()

In [ ]:
df.info()

---
## 2. What is the distribution of your most important column?

**Choice:** **`rating`** (1–5) is the main outcome for review data — it tells you how satisfied users are.

**Answer:** The counts below show how many reviews fall in each rating. `normalize=True` adds proportions (0–1).

In [ ]:
rating_dist = df["rating"].value_counts().sort_index()
rating_pct = df["rating"].value_counts(normalize=True).sort_index().mul(100).round(1)
summary = pd.DataFrame({"count": rating_dist, "percent": rating_pct})
summary

In [ ]:
print("Mean rating:", round(df["rating"].mean(), 2))
print("Median rating:", df["rating"].median())
print(
    "Interpretation: ratings skew high — many 4s and 5s — so the sample is not evenly spread across 1–5."
)

---
## 3. Filter to a meaningful subset. What is in it?

**Subset:** reviews with **`rating` ≤ 2** (low scores). These rows are the strongest negative signal for product issues.

**Answer:** We report how many rows match, show a sample, and one quick numeric summary (`mean` helpful votes in that slice).

In [ ]:
low = df[df["rating"] <= 2]
print(f"Subset size: {len(low)} rows ({len(low) / len(df):.1%} of all reviews)")
low[["app", "category", "rating", "verified_purchase", "device_type", "review"]].head(8)

In [ ]:
print("Average helpful_votes in low-rating subset:", round(low["helpful_votes"].mean(), 2))
print("Average helpful_votes overall:", round(df["helpful_votes"].mean(), 2))

---
## 4. Group by a category and find the average of a numeric column

**Grouping:** **`category`** (product type). **Numeric column:** **`rating`** (also show **`helpful_votes`** for context).

**Answer:** Average rating (and helpful votes) per category — easy comparison across product types.

In [ ]:
by_cat = df.groupby("category", observed=True)[["rating", "helpful_votes"]].mean().round(3)
by_cat = by_cat.sort_values("rating", ascending=False)
by_cat

---
## 5. Where are the missing values? Are any columns incomplete?

**Answer:** Count missing per column, then express as a percentage of rows. Any column with count **> 0** is incomplete for at least some reviews.

In [ ]:
miss = df.isnull().sum()
miss_pct = (df.isnull().mean() * 100).round(2)
missing_report = pd.DataFrame({"missing_count": miss, "missing_pct": miss_pct})
missing_report[missing_report["missing_count"] > 0]

In [ ]:
incomplete = missing_report[missing_report["missing_count"] > 0].index.tolist()
if incomplete:
    print("Columns with at least one missing value:", ", ".join(incomplete))
else:
    print("No missing values in any column.")